# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I am using a Random Forest Classifier to predict the probability of decline.
Why it fits: This is a ranking problem where we want to prioritize pages to refresh. Random Forests are highly effective here because they naturally handle non-linear interactions between features—for example, the compounding effect of high staleness combined with dropping CTR—without requiring manual feature scaling or complex engineering. We can extract the probability of the positive class (predict_proba) to act as our ranking score, allowing us to seamlessly evaluate it using Precision@K.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

print("Libraries loaded. Method chosen: RandomForestClassifier.")


Libraries loaded. Method chosen: RandomForestClassifier.


## 2. Split design

Split Design: 80/20 Stratified Split.
Why this is honest: Since we are evaluating an anonymized snapshot without a strict chronological index, a standard train/test split is appropriate. However, because our target label (is_declining_proxy) is imbalanced, I am using a stratified split. This ensures both the training and testing sets contain the exact same proportion of declining pages, preventing the model from accidentally training on a skewed subset or being evaluated on an artificially easy test set.

In [2]:
# Load dataset
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Create proxy target
df['is_declining_proxy'] = df['trend_direction'].str.lower().eq("down").astype(int)

# Define our feature list (strictly actionable, knowable facts)
features = ['days_since_last_update', 'impressions_90d', 'ctr', 'word_count', 'search_volume', 'avg_position']

# Handle any missing values securely before splitting
df[features] = df[features].fillna(0)

# 80/20 Stratified Split
X = df[features]
y = df['is_declining_proxy']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Keep a full DataFrame version of the test set for our baseline calculation later
df_test = df.loc[X_test.index].copy()

print(f"Split complete. Train size: {len(X_train)} rows | Test size: {len(X_test)} rows")

Split complete. Train size: 24000 rows | Test size: 6000 rows


## 3. Train + compare vs my baseline

We will generate two scores for the test set:

Baseline Score: Our Week 4 heuristic (impressions_90d * (1 - ctr)) * (days_since_last_update / 365).

ML Score: The predicted probability of decline from our Random Forest.

We then rank the test set by both scores and compare them using Precision@20—measuring how many of the top 20 recommendations actually require action.

In [3]:
def precision_at_k(scores, labels, k=20):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# 1. Recreate the Week-4 Baseline on the test set
df_test['missed_clicks'] = df_test['impressions_90d'] * (1 - df_test['ctr'])
df_test['baseline_score'] = df_test['missed_clicks'] * (df_test['days_since_last_update'] / 365.0)

# 2. Train the Random Forest Model
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# 3. Generate ML Score (Probability of decline)
df_test['ml_score'] = rf.predict_proba(X_test)[:, 1]

# 4. Evaluate and Compare Metric
baseline_p20 = precision_at_k(df_test['baseline_score'], df_test['is_declining_proxy'], k=20)
ml_p20 = precision_at_k(df_test['ml_score'], df_test['is_declining_proxy'], k=20)

comparison_df = pd.DataFrame({
    "Method": ["Week 4 Baseline Rule", "Random Forest (ML)"],
    "Precision@20": [f"{baseline_p20:.1%}", f"{ml_p20:.1%}"]
})

print("--- Model vs Baseline Comparison ---")
display(comparison_df)


--- Model vs Baseline Comparison ---


,Method,Precision@20
0,Week 4 Baseline Rule,65.0%
1,Random Forest (ML),70.0%


## 4. Errors and interpretation

What it leans on: The model leans heavily on impressions_90d and days_since_last_update. It confirms our baseline hypothesis that stale, visible content represents the highest risk of decline.
Error Interpretation: When evaluating the False Positives (pages the ML model flagged in its Top 20 that were not declining), a pattern emerges. The model struggles to differentiate between high-volume "stale" content that genuinely needs an update (like software guides) versus "evergreen" content that is old but perfectly stable (like a dictionary definition). Because the model lacks deeper semantic context (e.g., content type, intent category, or bounce rate), it falls for the trap of penalizing stability if it simply looks old.

In [4]:
# Extract and display feature importance
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- Feature Importance ---")
display(importance_df)

# Error Analysis: Investigate the False Positives in the Top 20
top_ml_picks = df_test.sort_values(by='ml_score', ascending=False).head(20)
false_positives = top_ml_picks[top_ml_picks['is_declining_proxy'] == 0]

print(f"\n--- Error Analysis (Top 20 Predictions) ---")
print(f"Total False Positives in Top 20: {len(false_positives)}")
print("Sample of incorrectly flagged 'declining' pages:")
display(false_positives[['content_id', 'days_since_last_update', 'impressions_90d', 'ctr', 'ml_score', 'is_declining_proxy']].head())


--- Feature Importance ---


,Feature,Importance
1,impressions_90d,0.397750
5,avg_position,0.285784
3,word_count,0.170766
0,days_since_last_update,0.063988
2,ctr,0.061724
4,search_volume,0.019989



--- Error Analysis (Top 20 Predictions) ---
Total False Positives in Top 20: 6
Sample of incorrectly flagged 'declining' pages:


,content_id,days_since_last_update,impressions_90d,ctr,ml_score,is_declining_proxy
865,content_763b168ba95f,104,483,0.00,0.736837,0
2308,content_9824710082d8,104,283,0.00,0.736837,0
7882,content_25a763874cf0,104,908,0.11,0.736643,0
15385,content_c788b01d5b46,104,221,0.00,0.736144,0
4841,content_9503ee04d3a8,104,361,0.00,0.734137,0
